## Setup

In [1]:
import os
from dotenv import load_dotenv
import psycopg2

load_dotenv()
PGHOST=os.getenv('PGHOST')
PGPORT=os.getenv('PGPORT')
PGDATABASE=os.getenv('PGDATABASE')
PGUSER=os.getenv('PGUSER')
PGPASSWORD=os.getenv('PGPASSWORD')
assert PGPASSWORD is not None


## Database Connection

In [2]:
from sqlalchemy import create_engine, URL
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=PGUSER,
    password=PGPASSWORD,
    host=PGHOST,
    port=int(PGPORT) if  PGPORT else 5432,
    database=PGDATABASE,
)
engine = create_engine(connection_url)

## Load the cohort

In [3]:
import pandas as pd

df = pd.read_sql("SELECT * FROM analysis.cohort", engine)


## First look

In [4]:
print(df.shape)
display(df.head())
df["bucket"].value_counts()

(47210, 9)


,nct_id,status_2018,status_2026,bucket,label,label_sensitivity,registration_date,registration_year,split
0,NCT03760900,NOT_YET_RECRUITING,COMPLETED,label 0: completed,0.0,0.0,2018-11-23,2018,test
1,NCT03760887,RECRUITING,COMPLETED,label 0: completed,0.0,0.0,2018-11-26,2018,test
2,NCT03760861,NOT_YET_RECRUITING,UNKNOWN,excluded: unknown status,NaN,1.0,2018-04-26,2018,test
3,NCT03760835,RECRUITING,RECRUITING,excluded: still ongoing,NaN,NaN,2018-11-09,2018,test
4,NCT03760822,RECRUITING,UNKNOWN,excluded: unknown status,NaN,1.0,2018-07-17,2018,test


bucket
label 0: completed             25420
excluded: unknown status       11326
label 1: terminated             4846
excluded: still ongoing         3701
label 1: withdrawn              1637
excluded: suspended              207
excluded: not found in 2026       63
excluded: other                   10
Name: count, dtype: int64

## Stopped-early rate

In [5]:
print(df["label"].mean())

0.20320972949252422


## Transition table

In [6]:
((pd.crosstab(df["status_2018"],df["status_2026"].fillna('NOT_FOUND_IN_2026'),normalize="index"))*100).round(1).T

status_2018,NOT_YET_RECRUITING,RECRUITING
status_2026,,
ACTIVE_NOT_RECRUITING,3.4,5.1
APPROVED_FOR_MARKETING,0.0,0.0
AVAILABLE,0.0,0.0
COMPLETED,37.9,58.2
ENROLLING_BY_INVITATION,0.1,0.1
NOT_FOUND_IN_2026,0.2,0.1
NOT_YET_RECRUITING,1.0,0.1
NO_LONGER_AVAILABLE,0.0,0.0
RECRUITING,2.8,2.8


## Split sizes

In [7]:
labeled=df[df['label'].notna()]
labeled.groupby('split')['label'].agg(['count', 'mean']).round(2)

,count,mean
split,,
test,10970,0.20
train,11776,0.21
valid,9157,0.20
